# Lesson 11: Prompt engineering

*Worked solutions for [the lesson page](../../lessons/11-prompt-engineering.html). The exercise version without solutions is `../11-prompt-engineering.ipynb`.*

Write prompts that work reliably, not just once, through clear instructions, context, examples and structure, along with reasoning and chaining techniques, and treat prompts as tested, versioned code.

**Runs on:** Any computer, plus an API key.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()   # reads ANTHROPIC_API_KEY from a .env file next to this notebook, if there is one
# On Google Colab, add the key under the key icon (Secrets), then uncomment the next two lines:
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"          # check your provider's current model list
print("API key found" if os.environ.get("ANTHROPIC_API_KEY") else
      "No API key set: cells marked 'Needs an API key' will fail until you add one")

## Principles

### 1. Be explicit about the task, the audience and the output

Compare:

*Vague*

```text
Summarise this incident report.
```

*Specific*

```text
Summarise the incident report below for the on-call engineering manager, who will
decide within five minutes whether to wake up the hardware team.

Write at most 5 bullet points:
- what failed, where, and since when
- current impact on data collection
- what has already been tried
- the single most likely cause, if the report supports one
- the decision needed, phrased as a yes/no question

If the report does not say something, write "not stated" rather than guessing.
```

The second prompt says who the reader is, why they need it, the format, the length, and what to do when information is missing. Each of those removes a guess.

### 4. Structure the prompt

Long prompts mix instructions, reference material and input data. Separate them clearly so the model never confuses data with instructions. XML-style tags work well and are easy to parse back out:

```text
<instructions>
Classify the support email into exactly one category and explain briefly.
</instructions>

<categories>
hardware: physical faults, power, cables, sensors, enclosures
software: firmware, drivers, the acquisition application, updates
data: missing, corrupted or suspicious measurements, processing results
other: anything else, including billing and scheduling
</categories>

<email>
{email_text}
</email>

Answer in the form:
<category>...</category>
<reason>one sentence</reason>
```

### Hands-on: a prompt linter

Turn the five principles into a checklist your code can run. It will not catch everything, and that is fine: its job is to catch the omissions you make when in a hurry.

1. Write `lint_prompt(text: str) -> list[str]` that returns one short message per problem found: no audience stated; no output format stated; no length limit; no rule for missing information; an instruction phrased only as a prohibition ("don't", "never") with no positive alternative nearby; an XML-style tag opened but not closed; and a `{placeholder}` for user input that is not wrapped in tags. Simple keyword and regular-expression checks are enough.
2. Run it on the vague and the specific incident-report prompts above, and on the classification prompt with the `<email>` tags.
3. Write three prompts for tasks you actually do, lint them, and rewrite until clean.
4. You are done when the vague prompt produces at least four findings, the specific one produces none, and you can explain one thing the linter missed in your own prompts that a reader would have caught.

#### Solution

In [ ]:
import re

def lint_prompt(text: str) -> list[str]:
    low = text.lower()
    problems = []
    if not re.search(r"\bfor (the|a|an|our|your)\b|\breader|\baudience|\bwho will\b", low):
        problems.append("no audience stated")
    if not re.search(r"bullet|json|table|in the form|format|sentence|line|paragraph|markdown|csv", low):
        problems.append("no output format stated")
    if not re.search(r"\b(at most|no more than|maximum|up to|exactly|one|two|three|\d+)\b.{0,20}\b(words|sentences|bullet|lines|points|items)", low):
        problems.append("no length limit")
    if not re.search(r"not stated|unknown|if .* (does not|doesn't|is missing)|say so|cannot be determined", low):
        problems.append("no rule for missing information")
    for line in low.splitlines():
        if re.match(r"\s*(don'?t|do not|never)\b", line) and not re.search(r"instead|rather|prefer|use ", line):
            problems.append(f"prohibition without a positive alternative: {line.strip()[:50]!r}")
    for tag in set(re.findall(r"<([a-z_]+)>", low)):
        if low.count(f"<{tag}>") != low.count(f"</{tag}>"):
            problems.append(f"tag <{tag}> is not closed")
    for ph in re.findall(r"\{(\w+)\}", text):
        if not re.search(r"<(\w+)>\s*\{" + ph + r"\}\s*</\1>", text):
            problems.append(f"placeholder {{{ph}}} is not delimited by tags")
    return problems

vague = "Summarise this incident report."
specific = """Summarise the incident report below for the on-call engineering manager, who will
decide within five minutes whether to wake up the hardware team.
Write at most 5 bullet points: what failed, where, and since when; current impact;
what has already been tried; the single most likely cause; the decision needed.
If the report does not say something, write "not stated" rather than guessing."""
print(lint_prompt(vague))
print(lint_prompt(specific))
print(lint_prompt("Classify the email.\n<email>\n{email_text}\n</email>\nDon't be verbose."))

The vague prompt gets four findings; the specific one none; the last one is flagged for the bare prohibition and for having no audience, format, length or missing-information rule. The linter is deliberately crude: a prompt can pass it and still be unclear, which is why the real test is the labelled set in the last session.

## Techniques

### Few-shot prompting

Include worked input-output pairs. In a chat API you can write them as earlier conversation turns, which models treat as strong evidence of the desired behaviour.

In [ ]:
new_reading = "O3 reads exactly 0.0 ppb for ten minutes every day at 12:00"
messages = [
    {"role": "user", "content": "Reading: 'NO2 412 ppb at 03:00, all other hours 10-25 ppb'"},
    {"role": "assistant", "content": "SUSPECT: isolated spike roughly 20x the surrounding values."},
    {"role": "user", "content": "Reading: 'NO2 rises from 15 to 38 ppb between 07:00 and 09:00'"},
    {"role": "assistant", "content": "PLAUSIBLE: gradual morning increase consistent with traffic."},
    {"role": "user", "content": f"Reading: '{new_reading}'"},
]

In [ ]:
# Needs an API key (see the setup cell)
r = client.messages.create(model=MODEL, max_tokens=500, messages=messages)
print("".join(b.text for b in r.content if b.type == "text"))

### Let the model reason before answering

For multi-step problems, asking the model to work through the problem before giving the final answer improves accuracy, because each generated token is extra computation the model can build on. This is **chain-of-thought** prompting. Ask for the reasoning in a separate section so your code can discard it and keep only the answer:

```text
Work out the answer step by step inside <thinking> tags.
Then give only the final number inside <answer> tags.
```

**Reasoning models** (lesson 12) do this internally, controlled by an effort setting. With them, "think step by step" is redundant, and prescribing the steps in detail can hurt: give the goal, the constraints and what a correct answer looks like.

### Prompt chaining

One giant prompt that researches, drafts, critiques and formats is hard to debug. Break the job into steps, each a separate call with a focused prompt, passing outputs forward. A one-line helper keeps the chain readable:

In [ ]:
def llm(prompt: str, max_tokens: int = 2000) -> str:
    """One prompt in, the reply text out."""
    r = client.messages.create(model=MODEL, max_tokens=max_tokens,
                               messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in r.content if b.type == "text")

In [ ]:
def answer_with_review(question: str, document: str, llm=llm) -> str:
    quotes = llm(f"<doc>{document}</doc>\nList verbatim quotes relevant to: {question}")
    draft = llm(f"<quotes>{quotes}</quotes>\nAnswer '{question}' using only these quotes, citing them.")
    review = llm(f"<quotes>{quotes}</quotes>\n<answer>{draft}</answer>\n"
                 "List any claim in the answer not supported by the quotes, or reply NONE.")
    if review.strip() != "NONE":
        draft = llm(f"Rewrite the answer removing unsupported claims.\n<problems>{review}</problems>\n<answer>{draft}</answer>")
    return draft

In [ ]:
# Needs an API key (see the setup cell)
doc = ("Replace the desiccant cartridge every six months, or every three months at humid sites. "
       "The indicator window turns pink when the desiccant is saturated.")
print(answer_with_review("How often should the desiccant be replaced at a humid site?", doc))

### Self-consistency and verification

For questions with a checkable answer, sample several responses and take the majority answer. Model outputs vary from call to call, so disagreements are informative: when four of five samples agree, the answer is probably right; when they split, the question is hard or ambiguous. Lesson 12 treats this systematically, including when voting helps and when it only multiplies the bill.

### Hands-on: self-consistency with a vote

Build the sampling-and-voting loop, with the parsing that makes it robust, and test it on a scripted model first.

1. Write `extract_answer(text) -> str | None` that returns the content of the last `<answer>` tag, stripped, or `None` if there is none.
2. Write `vote(question, llm, n=5) -> tuple[str | None, float]` that asks `n` times with the chain-of-thought instruction above, extracts the answers, drops the `None`s, and returns the most common answer and its share of the valid votes.
3. Write `scripted(replies)` that returns an `llm` handing out the replies in turn, and script five: three that end in `<answer>32</answer>`, one with `<answer>30</answer>`, and one with no tag at all.
4. You are done when the scripted vote returns `("32", 0.75)`, and, with a key, running the real model five times on "A sensor logs a reading every 15 seconds; 5% fail the quality check. How many good readings in 6 hours?" gives a vote you can explain (the answer is 1368).

#### Solution

In [ ]:
import re
from collections import Counter

COT = ("Work out the answer step by step inside <thinking> tags.\n"
       "Then give only the final number inside <answer> tags.\n\n")

def extract_answer(text: str) -> str | None:
    found = re.findall(r"<answer>(.*?)</answer>", text, flags=re.S)
    return found[-1].strip() if found else None

def vote(question: str, llm, n: int = 5):
    answers = [extract_answer(llm(COT + question)) for _ in range(n)]
    valid = [a for a in answers if a is not None]
    if not valid:
        return None, 0.0
    best, count = Counter(valid).most_common(1)[0]
    return best, count / len(valid)

def scripted(replies):
    replies = iter(list(replies))
    return lambda prompt: next(replies)

fake = scripted(["<thinking>...</thinking><answer>32</answer>", "<answer>32</answer>",
                 "<thinking>...</thinking><answer>30</answer>", "I think it is 32.",
                 "<answer> 32 </answer>"])
print(vote("how many hours?", fake))          # ('32', 0.75)

# print(vote("A sensor logs a reading every 15 seconds; 5% fail the quality check. "
#            "How many good readings in 6 hours?", llm))      # Needs an API key

The missing-tag reply is dropped rather than counted, which matters: a parser that treated it as the answer "I think it is 32." would never match the other votes. On the real question, a capable model usually agrees with itself five times out of five (6 hours is 1,440 readings, 95% of which is 1,368), so the vote buys nothing there; it pays on questions the model gets right only most of the time.

## Prompts as code

### Templates

Keep prompts in files or constants, separate from logic, and fill them with variables. Escape or delimit user input so it cannot break your structure.

In [ ]:
from string import Template

CLASSIFY_V3 = Template("""<instructions>
Classify the email into exactly one category: hardware, software, data or other.
</instructions>
<email>
$email
</email>
Reply with only the category name, in lower case.""")

def classify(email: str) -> str:
    prompt = CLASSIFY_V3.substitute(email=email.replace("</email>", ""))
    r = client.messages.create(model=MODEL, max_tokens=256,          # room for a little thinking
                               output_config={"effort": "low"},       # a one-word answer needs no deep thought
                               messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in r.content if b.type == "text").strip().lower()

### Test prompts against examples, not vibes

A prompt that looks great on three examples you tried can fail on the fourth. Build a small labelled set (even 20 to 50 cases, including tricky ones), run every prompt version against it, and track the score. This is the seed of the evaluation discipline in lesson 17, and the single habit that most separates professionals from tinkerers.

In [ ]:
cases = [
    ("The fan in the enclosure is grinding loudly", "hardware"),
    ("After updating to 2.4 the GUI crashes on start", "software"),
    ("Yesterday's ozone column values are all exactly zero", "data"),
    ("Can we move the calibration visit to March?", "other"),
    # ... at least 20, including ambiguous ones
]

def score(classify_fn):
    results = [(text, gold, classify_fn(text)) for text, gold in cases]
    acc = sum(g == p for _, g, p in results) / len(results)
    for text, gold, pred in results:
        if gold != pred:
            print(f"MISS  gold={gold:<9} pred={pred:<9} {text}")
    return acc

In [ ]:
# Needs an API key (see the setup cell)
print("accuracy:", score(classify))

### Automatic prompt optimisation

Once you have a labelled set and a scoring function, the improvement loop can be automated: an optimiser proposes instruction variants and few-shot selections, scores each on your set, and keeps the best (DSPy popularised this). It can only be as good as your metric and your labelled set, and it will overfit to both, so hold out cases it never sees.

### Hands-on: iterate a classifier prompt to 90%+

1. Write 30 short support messages across the four categories above. Make at least 8 deliberately ambiguous ones (a firmware update that bricked a sensor; a data gap caused by a power cut).
2. Write the simplest possible classification prompt and score it with `score()`.
3. Iterate: add category definitions, add tie-break rules for the ambiguous cases ("classify by root cause if stated, otherwise by symptom"), add two or three examples. Score each version and keep a table in your log.
4. Try a smaller, cheaper model with your best prompt. Is the accuracy loss acceptable for the cost saving?

No key? Run the same harness against a keyword-rule classifier you write yourself and iterate *its* rules on the misses. The habit of scoring every version on the same set is the point; the model is interchangeable.

#### Solution: what usually happens

A typical log looks something like this:

| Version | Change | Accuracy |
|---|---|---|
| v1 | "Classify into hardware, software, data or other" | 70% |
| v2 | Added one-line definitions of each category | 80% |
| v3 | Added tie-break rule: root cause if stated, else the symptom the user reports | 90% |
| v4 | Added three diverse examples, including one ambiguous case | 93% |
| v4 on small model | Same prompt, cheaper model | 90% |

Your numbers will differ. What matters is the pattern: the biggest jumps come from **removing ambiguity** (definitions, tie-break rules), not from clever wording, and the ambiguous cases decide the final score. The remaining misses are often cases where *you* are not sure of the right label either, which is a sign to clarify the labelling policy itself. The small-model result is a business decision: if 3 points of accuracy cost a fifth of the price, many teams take it, perhaps routing low-confidence cases to the larger model.

In [ ]:
# A keyword-rule classifier for the no-key route: iterate its rules on the misses with score().
RULES = {"hardware": ["fan", "cable", "power", "sensor", "enclosure", "belt"],
         "software": ["update", "firmware", "driver", "crash", "gui", "app"],
         "data": ["values", "zero", "missing", "gap", "corrupt", "ozone"],
         "other": ["invoice", "visit", "schedule", "access", "move"]}

def classify_rules(email: str) -> str:
    low = email.lower()
    hits = {cat: sum(w in low for w in words) for cat, words in RULES.items()}
    best = max(hits, key=hits.get)
    return best if hits[best] else "other"

print("accuracy:", score(classify_rules))

## Check yourself

Take the quiz at the end of the lesson page. Then close everything and write three or four sentences in your learning log about what you learned, from memory.